In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
amazon_labelled = pd.read_csv('sentiment labelled sentences\\amazon_cells_labelled.txt', sep='\t', names=['text', 'label'])
print(amazon_labelled.shape)
print(amazon_labelled.head())

(1000, 2)
                                                text  label
0  So there is no way for me to plug it in here i...      0
1                        Good case, Excellent value.      1
2                             Great for the jawbone.      1
3  Tied to charger for conversations lasting more...      0
4                                  The mic is great.      1


In [3]:
amazon_labelled['source'] = 'amazon'
print(amazon_labelled.head())

                                                text  label  source
0  So there is no way for me to plug it in here i...      0  amazon
1                        Good case, Excellent value.      1  amazon
2                             Great for the jawbone.      1  amazon
3  Tied to charger for conversations lasting more...      0  amazon
4                                  The mic is great.      1  amazon


In [4]:
imdb_labelled = pd.read_csv('sentiment labelled sentences\\imdb_labelled.txt', sep='\t', names=['text', 'label'], quoting=3)
print(imdb_labelled.shape)
print(imdb_labelled.head())
print(imdb_labelled.label.value_counts())

(1000, 2)
                                                text  label
0  A very, very, very slow-moving, aimless movie ...      0
1  Not sure who was more lost - the flat characte...      0
2  Attempting artiness with black & white and cle...      0
3       Very little music or anything to speak of.        0
4  The best scene in the movie was when Gerardo i...      1
label
0    500
1    500
Name: count, dtype: int64


In [5]:
imdb_labelled['source'] = 'imdb'
print(imdb_labelled.head())

                                                text  label source
0  A very, very, very slow-moving, aimless movie ...      0   imdb
1  Not sure who was more lost - the flat characte...      0   imdb
2  Attempting artiness with black & white and cle...      0   imdb
3       Very little music or anything to speak of.        0   imdb
4  The best scene in the movie was when Gerardo i...      1   imdb


In [6]:
yelp_labelled = pd.read_csv('sentiment labelled sentences\\yelp_labelled.txt', sep='\t', names=['text', 'label'])
print(yelp_labelled.shape)
print(yelp_labelled.head())

(1000, 2)
                                                text  label
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


In [7]:
yelp_labelled['source'] = 'yelp'
print(yelp_labelled.head())

                                                text  label source
0                           Wow... Loved this place.      1   yelp
1                                 Crust is not good.      0   yelp
2          Not tasty and the texture was just nasty.      0   yelp
3  Stopped by during the late May bank holiday of...      1   yelp
4  The selection on the menu was great and so wer...      1   yelp


In [8]:
data = pd.concat([amazon_labelled, imdb_labelled, yelp_labelled], ignore_index=True)

In [9]:
print('shape', data.shape)
print('count of NAN:\n', data.isna().sum())
print('distribution\n', data.groupby(['source', 'label']).size())

shape (3000, 3)
count of NAN:
 text      0
label     0
source    0
dtype: int64
distribution
 source  label
amazon  0        500
        1        500
imdb    0        500
        1        500
yelp    0        500
        1        500
dtype: int64


In [10]:
duplicates_text = data.duplicated(subset='text')
duplicates_all = data.duplicated(subset=['text', 'label'])
text_indexes = list(duplicates_text[duplicates_text == True].index)
all_indexes = list(duplicates_all[duplicates_all == True].index)

print(text_indexes == all_indexes)

True


In [11]:
data = data.drop_duplicates(subset=['text'], ignore_index=True)

print('shape', data.shape)
print(f'distribution\n{data.groupby(['source', 'label']).size()}')

shape (2983, 3)
distribution
source  label
amazon  0        497
        1        493
imdb    0        499
        1        498
yelp    0        497
        1        499
dtype: int64


In [12]:
stratify_tt = data.source + '_' + data.label.astype('str')
X_trainval, X_test, y_trainval, y_test = train_test_split(
    data.text, data.label,
    test_size=0.15,
    random_state=42,
    stratify=stratify_tt
)


stratify_tv = data.loc[X_trainval.index].source + '_' + data.loc[X_trainval.index].label.astype('str')
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.18,
    random_state=42,
    stratify=stratify_tv
)

print(f'X_train shape: {X_train.shape}\nX_val shape: {X_val.shape}\nX_test shape: {X_test.shape}')
print(data.loc[X_test.index].groupby(['source', 'label']).size())
print(data.loc[X_train.index].groupby(['source', 'label']).size())
print(data.loc[X_val.index].groupby(['source', 'label']).size())

X_train shape: (2078,)
X_val shape: (457,)
X_test shape: (448,)
source  label
amazon  0        75
        1        74
imdb    0        75
        1        75
yelp    0        74
        1        75
dtype: int64
source  label
amazon  0        346
        1        343
imdb    0        348
        1        347
yelp    0        347
        1        347
dtype: int64
source  label
amazon  0        76
        1        76
imdb    0        76
        1        76
yelp    0        76
        1        77
dtype: int64


Разделение сохранило состав данных по источникам и классам.